## **Problem Statement**

#### **What is the problem I am trying to solve?**

Human emotions play a very important role in understanding a person’s mental and emotional state.
In real life, emotions are not expressed only through words — they are strongly reflected through facial expressions.

The goal of this module is to detect the emotion of a user from a single facial image.

This CNN model is not the final product itself — it is a sub-module of a larger multimodal system called EmoNarrator, which combines:

- image understanding (CNN)

- contextual reasoning (LLM)

- voice narration (TTS)

This notebook focuses exclusively on the **Large Language Model (LLM)**
component of the system.

The responsibility of this LLM is not emotion detection, but emotional
interpretation and narration generation.

This notebook explains, step by step, how a pretrained language model
was fine-tuned to generate meaningful emotional narrations based on
emotion signals extracted by a CNN model.



## **NOTEBOOK OBJECTIVE**

## 🎯 Objective of This Notebook

The objective of this notebook is to:

- Fine-tune a pretrained GPT-based language model
- Teach it how to interpret emotional signals
- Generate calm, human-like emotional narration
- Align narration behavior with CNN emotion outputs

At the end of this notebook, we will obtain:

- An emotion-aware narration model
- A deployable Hugging Face model
- A reusable language module for the Emotionary pipeline


1 — Generate a bootstrap dataset (500 examples)

Save as make_dataset.py. This uses KDEF templates and synthetic metadata to create a training JSONL. You can later replace data/kdef_train.jsonl with the file produced by the kdef_to_jsonl.py that uses actual CNN probs.

## **MODEL SELECTION**

We use DistilGPT-2 as the base language model.

DistilGPT-2 is a lightweight version of GPT-2 that retains strong
language generation capabilities while being computationally efficient.

Reasons for choosing DistilGPT-2:
- Pretrained on large English corpora
- Suitable for small dataset fine-tuning
- Faster inference
- Ideal for real-time narration systems

Instead of training a model from scratch, we adapt an existing
foundation model to our emotional narration task.

In [ ]:
#installing dependicies
!pip install transformers datasets accelerate torch


## **Dataset Description**

In [ ]:
#Creating dataset containing 500 examples for fine tune My base llm model
import json, random, os
os.makedirs("data", exist_ok=True)

TEMPLATES = {
 "sad": [
   "At {time}, the person in the image looks quietly sad, as if lost in thought rather than visibly upset. There’s a softness to their expression suggesting low energy rather than anger. It reads like a private pause between activities, where motivation feels thin. The moment feels ordinary but heavier than the setting would suggest. This sadness seems likely to pass rather than be acute."
 ],
 "happy": [
   "At {time}, the person shows a gentle happiness — eyes lifted slightly and a relaxed jaw. The expression reads as a grounded little joy, perhaps from a pleasant interaction or small success. The scene is casual and the mood feels light rather than ecstatic. It’s a pleasant, composed moment."
 ],
 "angry": [
   "At {time}, the person looks tense, with a sharpness in the jaw and eyes that suggests frustration more than explosive anger. It reads like a brief professional irritation or personal annoyance. The posture hints at held-back words. Treat this as an observation rather than a diagnosis."
 ],
 "surprise": [
   "At {time}, the person appears surprised — brows slightly raised and eyes widened. It feels like a momentary reaction to something unexpected, not a sustained state. The scene looks immediate and passing."
 ],
 "neutral":[
   "At {time}, the expression is calm and neutral; little sign of strong emotion. The person seems composed and at ease, and the scene reads as ordinary. There is no clear evidence of distress or elation."
 ],
 "fear":[
   "At {time}, the eyes and mouth suggest unease or worry. The expression reads more cautious than panicked. It could be a reaction to a small trigger or general concern. This is descriptive and not diagnostic."
 ]
}

times = ["08:00","09:30","12:00","14:00","16:30","19:00"]
ages = [18,20,22,24,27,30,35,40]
professions = ["student","barista","teacher","shopkeeper","developer","engineer","doctor"]
envs = ["classroom","coffee shop","office","home","train","park"]

def make_example():
    e = random.choice(list(TEMPLATES.keys()))
    t = random.choice(times)
    age = random.choice(ages)
    prof = random.choice(professions)
    env = random.choice(envs)
    # synthetic confidence varied to teach hedging
    conf = round(random.uniform(0.2, 0.95), 2) if random.random() < 0.15 else round(random.uniform(0.6, 0.95),2)
    prompt = (f"Time: {t}\nAge: {age}\nProfession: {prof}\nEnvironment: {env}\n"
              f"Dominant emotion: {e}\nConfidence: {conf}\nExtra hint: none\n\n"
              "Write a 4-6 sentence empathetic narration in British English. Do NOT diagnose; if confidence < 0.4 start with 'Possibly'.")
    out = random.choice(TEMPLATES[e]).format(time=t)
    return {"input": prompt, "output": out}

# create dataset
N = 500  # change to 1000–2000 if you want more and can wait longer on CPU training
with open("data/train_small.jsonl", "w", encoding="utf-8") as f:
    for _ in range(N):
        ex = make_example()
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Wrote data/train_small.jsonl with", N, "examples")


Wrote data/train_small.jsonl with 500 examples


The dataset used for fine-tuning is custom-built for Emotionary.

Each training example contains two parts:

### Input (Prompt)
- Emotion label
- Emotion probability
- Additional user metadata
  - profession
  - time of day
  - environment

### Output (Completion)
- Emotionally grounded narration
- Human-like descriptive explanation
- Calm and empathetic language style

This dataset does not teach emotion detection.
It teaches emotional interpretation.

2 — CPU-optimised fine-tune script (distilgpt2) with prompt-masking

Save as train_cpu_sft_masked.py. This masks the prompt tokens (loss = -100) so the model learns to generate the narration only. Settings chosen for CPU: small batch, 3 epochs default, low LR.

## **Fine Tuning the basemodel distilgpt2**

### 1. Importing Required Libraries

Fine-tuning a language model involves:

- data preprocessing

- tokenization

- loss masking

- training orchestration

In [ ]:
import os, json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

### 2. Defining Base Model and Training Configuration

In [ ]:
MODEL_NAME = os.environ.get("BASE_MODEL", "distilgpt2")

TRAIN_FILE = os.environ.get("TRAIN_FILE", "data/train_small.jsonl")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "cpu-sft-distil")
EPOCHS = int(os.environ.get("EPOCHS", 3))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 1))
LR = float(os.environ.get("LR", 2e-5))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", 256))


### 3. Printing Training Configuration

In [ ]:
print(
    "Training config:",
    MODEL_NAME,
    TRAIN_FILE,
    OUTPUT_DIR,
    "epochs",
    EPOCHS,
    "batch",
    BATCH_SIZE
)

Training config: distilgpt2 data/train_small.jsonl cpu-sft-distil epochs 3 batch 1


### 4. Loading Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


### 5. Adding Padding Token

GPT-based models do not come with a padding token by default.

However, padding is required for:

- batching

- fixed-length inputs

- efficient training

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})


### 6. Loading Pretrained Language Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50258, 768)

### 7. Loading Training Dataset

Load the supervised fine-tuning dataset from JSONL format.

Each sample contains:

- input → emotional/context prompt

- output → narration text

In [ ]:
raw = load_dataset("json", data_files=TRAIN_FILE, split="train")


Generating train split: 0 examples [00:00, ? examples/s]

### 8. Designing the Preprocessing Function

his is the core intelligence of  training pipeline.

Here we define how the model learns.

What happens conceptually

We want the model to:

1. read the prompt

2. generate the reply

3. but NOT be punished for prompt tokens

4. Padding and Truncation: All inputs must have the same length for batching.Long samples → truncated, Short samples → padded

So we mask prompt tokens from loss.

This is Supervised Fine-Tuning (SFT).

In [ ]:
def preprocess(example):
    prompt = example["input"].strip() + tokenizer.eos_token
    reply = example["output"].strip() + tokenizer.eos_token
    # encode separately so we can mask prompt tokens from loss
    enc_prompt = tokenizer(prompt, add_special_tokens=False)
    enc_reply = tokenizer(reply, add_special_tokens=False)
    input_ids = enc_prompt["input_ids"] + enc_reply["input_ids"]
    attn = [1]*len(input_ids)
    labels = [-100]*len(enc_prompt["input_ids"]) + enc_reply["input_ids"]  # mask prompt tokens
    # pad/truncate to MAX_LENGTH
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attn = attn[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    else:
        pad_len = MAX_LENGTH - len(input_ids)
        input_ids = input_ids + [tokenizer.pad_token_id]*pad_len
        attn = attn + [0]*pad_len
        labels = labels + [-100]*pad_len
    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

### 9. Applying Preprocessing to Dataset


Convert raw JSON data into tokenized tensors usable .

In [ ]:
tokenized = raw.map(preprocess, remove_columns=raw.column_names, batched=False)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


### 10. Training Configuration

Define how training will run on CPU.

Key points:

- CPU training → fp16 disabled

- small batch size

- limited checkpoints

- clean logging

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=False,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
)


### 11. Trainer Initialization

The Trainer class manages:

- forward pass

- loss computation

- optimization

- checkpoint saving

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
print("Saved model to", OUTPUT_DIR)


Training config: distilgpt2 data/train_small.jsonl cpu-sft-distil epochs 3 batch 1


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,3.247300
100,1.411200
150,0.563200
200,0.262500
250,0.126800
300,0.069700
350,0.038100
400,0.029900
450,0.015200
500,0.010800


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Saved model to cpu-sft-distil


### **SAVE MODEL**

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilgpt2"              # base model you fine-tuned from
MODEL_DIR = "/content/cpu-sft-distil"  # your model folder

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.save_pretrained(MODEL_DIR)

print("Tokenizer saved into:", MODEL_DIR)


Tokenizer saved into: /content/cpu-sft-distil


### **Testing**

In [ ]:
# test_single.py
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_DIR = "/content/cpu-sft-distil"   # change if different
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)

prompt = (
    "Time: 12:00\n"
    "Age: 18\n"
    "Profession: student\n"
    "Environment: between classes\n"
    "Dominant emotion: sad\n"
    "Confidence: 0.78\n"
    "Extra hint: none\n\n"
    "Write a 4-6 sentence empathetic narration in British English. Do NOT diagnose; if confidence < 0.4 start with 'Possibly'.\n"
)

input_ids = tokenizer(prompt, return_tensors="pt").input_ids
out = model.generate(
    input_ids,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.95,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

text = tokenizer.decode(out[0], skip_special_tokens=False)
# Remove the prompt if the model echoes it
if text.startswith(prompt):
    narration = text[len(prompt):].strip()
else:
    # some tokenizers/outputs may include prompt with minor differences. Try split by newline
    narration = text.split("\n\n", 1)[-1].strip()

print("\n=== NARRATION ===\n")
print(narration)
print("\n=================\n")


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== NARRATION ===

At 12:00, the person in the image looks quietly sad, as if lost in thought rather than visibly upset. There’s a softness to their expression suggesting low energy rather than anger. It reads like a private pause between activities, where motivation feels thin. The moment feels ordinary but heavier than the setting would suggest. This sadness seems likely to pass rather than be acute.<|endoftext|>


